In [ ]:

import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import sys
sys.path.append("../src/")
from multiHIVE.model import multiHIVE
import torch
import numpy as np

In [2]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
torch.set_float32_matmul_precision("high")

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
adata = sc.read_h5ad( "/Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 25517 × 165454
    obs: 'cell_type', 'batch'
    var: 'modality'
    obsm: 'protein_counts'

In [5]:
# adata = scvi.data.organize_multiome_anndatas(adata)
adata = adata[:, adata.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata, min_cells=int(adata.shape[0] * 0.01))
multiHIVE.setup_anndata(adata, batch_key="batch", protein_expression_obsm_key = "protein_counts")
adata

INFO     Using column names from columns of adata.obsm['protein_counts']                                           


/tmp/ipykernel_1047203/2004131125.py:4: DeprecationWarning: multiHIVE is supposed to work with MuData. the use of anndata is deprecated and will be removed in scvi-tools 1.4. Please use setup_mudata
  multiHIVE.setup_anndata(adata, batch_key="batch", protein_expression_obsm_key = "protein_counts")


AnnData object with n_obs × n_vars = 25517 × 48872
    obs: 'cell_type', 'batch', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'modality', 'n_cells'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'protein_counts'

In [ ]:
vae = multiHIVE(adata, latent_distribution="normal",
                n_genes=(adata.var["modality"] == "Gene Expression").sum(),
                n_regions=(adata.var["modality"] == "Peaks").sum(),
                n_proteins=adata.obsm['protein_counts'].shape[1],
                mi_loss = True,
               )
vae.train()

In [7]:
vae.get_latent_representation()

In [ ]:
np.save("TEA-seq.npy", adata.obsm['Z_multiHIVE'])